# Callas — one clip at a time

Paste a **single YouTube link** → get the transcript with **who speaks to whom** (Speaker A / Speaker B turns). Pick clips yourself, run them one by one.

**Before running:**
1. Runtime → Change runtime type → **GPU** (T4).
2. **YouTube cookies** — Colab IPs hit *"Sign in to confirm you're not a bot"*, so YouTube downloads need a `cookies.txt` (see cookies cell). YouTube sometimes blocks Colab even with cookies — if so, use the **local audio** cell at the bottom.
3. For who-speaks-to-whom: a Hugging Face token + accepted terms for `pyannote/speaker-diarization-3.1` and `pyannote/segmentation-3.0`. (Skip → transcript only, one speaker.)

Run the setup cells once, then re-run a **run cell** for each new clip.

In [ ]:
!apt-get -qq install -y ffmpeg > /dev/null
!pip install -q "transformers<5" "accelerate>=0.30" "pyannote.audio>=3.1" pythainlp python-crfsuite webvtt-py pandas pyyaml tqdm
!pip install -q -U yt-dlp    # keep yt-dlp current — YouTube breaks it often
%cd /content
![ -d callas-dataset ] || git clone https://github.com/JoyChanya/callas-dataset.git
%cd /content/callas-dataset
!git pull -q
import torch; print('cuda', torch.cuda.is_available())

### YouTube cookies (for downloading on Colab)
1. In a browser logged in to **youtube.com**, export `cookies.txt` with a "Get cookies.txt LOCALLY" extension (Netscape format).
2. Run this cell → upload that file. It wires `youtube.cookiefile` into `config.yaml`.

Tip: use a throwaway Google account. Cookies expire — re-export if it starts failing again.

In [ ]:
from google.colab import files
up = files.upload()                      # choose your cookies.txt
name = next(iter(up))
import os; os.replace(name, 'cookies.txt')
!sed -i 's|cookiefile: ""|cookiefile: "cookies.txt"|' config.yaml
!grep cookiefile config.yaml

In [ ]:
# Hugging Face token (needed for who-speaks-to-whom). Enter to skip = transcript only.
import os, getpass
os.environ["HF_TOKEN"] = getpass.getpass("Paste HF token (or press Enter to skip): ")

### Run one clip (from a YouTube link)
Paste a link, run. Change the link and re-run for the next clip.

In [ ]:
URL = "https://www.youtube.com/watch?v=PASTE_ID_HERE"

!python -u process_one.py "{URL}" --device cuda --model biodatlab/whisper-th-large-v3-combined

### Fallback: a local audio file (when YouTube blocks Colab)
YouTube often refuses Colab's IP even with cookies. Reliable path: get the audio **off Colab** — on your own computer download it (yt-dlp works on a home connection), or use any YouTube→mp3 tool — then upload the file here. Accepts mp3 / m4a / wav / mp4.

In [ ]:
from google.colab import files
up = files.upload()                      # choose an audio/video file
AUDIO = next(iter(up))
!python -u process_one.py --audio "{AUDIO}" --device cuda --model biodatlab/whisper-th-large-v3-combined